# AutoGate router: train on Kaggle or Colab

Same steps as `scripts/kaggle_train.sh`: clone, install, generate the full dataset, train
Qwen3-0.6B + LoRA (3 epochs), calibrate, predict on test + ood, score, export ONNX + int8,
then the same for the `--no-context` ablation. Needs a **GPU (T4 or better)** and internet.
On Kaggle the work directory is `/kaggle/working`; on Colab it is `/content`. The repo is
public, so no token is needed. Run all cells; artifacts end up in `$WORK/runs`.


In [ ]:
import os

os.environ["REPO_URL"] = "https://github.com/anantha99/AutoGate.git"
os.environ["BRANCH"] = "claude/peaceful-pasteur-nessbd"
os.environ["EPOCHS"] = "3"
work = next((d for d in ("/kaggle/working", "/content") if os.path.isdir(d)), os.path.expanduser("~/autogate_work"))
os.makedirs(work, exist_ok=True)
os.environ["WORK"] = work
print("work dir:", work)
try:  # optional, only for a private fork
    from kaggle_secrets import UserSecretsClient

    os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass


In [ ]:
%%bash
set -euo pipefail
cd "$WORK"
url="$REPO_URL"
if [ -n "${GITHUB_TOKEN:-}" ]; then url="https://x-access-token:${GITHUB_TOKEN}@${REPO_URL#https://}"; fi
[ -d AutoGate/.git ] || git clone -q --depth 1 --branch "$BRANCH" "$url" AutoGate
cd AutoGate && git log --oneline -1
pip install -q -e ".[train,export]"
python -m autogate_bench.generate --out data/generated/full --rng-seed 0 --paraphrases data/paraphrases

In [ ]:
%%bash
set -euo pipefail
cd "$WORK/AutoGate"
ROWS=data/generated/full/rows.jsonl; RUNS="$WORK/runs"; mkdir -p $RUNS/rules
python -m autogate_eval.baselines.rules --rows $ROWS --emit $RUNS/rules/predictions.jsonl > $RUNS/rules/baseline_report.txt
python -m autogate_eval.score --rows $ROWS --preds $RUNS/rules/predictions.jsonl --split test,ood --json $RUNS/rules/score_test_ood.json

## Main run: context prefix + utterance

In [ ]:
%%bash
set -euo pipefail
cd "$WORK/AutoGate"
ROWS=data/generated/full/rows.jsonl; OUT="$WORK/runs/qwen3-0.6b"; mkdir -p $OUT
python -m autogate_router.train --config configs/train_qwen3_0.6b.yaml --rows $ROWS --out-dir $OUT --epochs $EPOCHS 2>&1 | tee $OUT/train.log
python -m autogate_router.calibrate --checkpoint $OUT/best --rows $ROWS --batch-size 128 | tee $OUT/calibrate.log
python -m autogate_router.predict --checkpoint $OUT/best --rows $ROWS --split test,ood --batch-size 128 --out $OUT/predictions_test_ood.jsonl
python -m autogate_eval.score --rows $ROWS --preds $OUT/predictions_test_ood.jsonl --split test,ood --json $OUT/score_test_ood.json | tee $OUT/score_test_ood.txt
python -m autogate_router.export --checkpoint $OUT/best --out-dir $OUT/onnx --rows $ROWS --split val --n-examples 256 2>&1 | tee $OUT/export.log

## Ablation: `--no-context` (utterance only)

In [ ]:
%%bash
set -euo pipefail
cd "$WORK/AutoGate"
ROWS=data/generated/full/rows.jsonl; OUT="$WORK/runs/qwen3-0.6b-no-context"; mkdir -p $OUT
python -m autogate_router.train --config configs/train_qwen3_0.6b.yaml --rows $ROWS --out-dir $OUT --epochs $EPOCHS --no-context 2>&1 | tee $OUT/train.log
python -m autogate_router.calibrate --checkpoint $OUT/best --rows $ROWS --batch-size 128 | tee $OUT/calibrate.log
python -m autogate_router.predict --checkpoint $OUT/best --rows $ROWS --split test,ood --batch-size 128 --out $OUT/predictions_test_ood.jsonl
python -m autogate_eval.score --rows $ROWS --preds $OUT/predictions_test_ood.jsonl --split test,ood --json $OUT/score_test_ood.json | tee $OUT/score_test_ood.txt
python -m autogate_router.export --checkpoint $OUT/best --out-dir $OUT/onnx --rows $ROWS --split val --n-examples 256 2>&1 | tee $OUT/export.log

In [ ]:
%%bash
cd "$WORK"/runs && find . -maxdepth 3 -not -path '*/tokenizer*' | sort && du -sh */onnx 2>/dev/null